# Heart Disease Prediction ML Project
### Exploratory Data Analysis & Machine Learning Training Pipeline

This notebook provides a step-by-step walkthrough of the **Heart Disease Prediction ML** project. We will explore patient clinical datasets, visualize key clinical correlations, preprocess features, train multiple classification models, compare their diagnostics performance, and select the best model to export.

## 1. Imports and Setup
First, let's import the core libraries and set up visualizations.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append src to path so we can import local modules
sys.path.append(os.path.abspath('../src'))

# Set visualization styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("Setup complete.")

## 2. Dataset Overview
Let's load the dataset from `../dataset/heart_disease.csv` and inspect its shape, columns, and target distribution.

In [ ]:
df = pd.read_csv('../dataset/heart_disease.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Let's check missing values in detail
print("Missing Values per Column:")
print(df.isnull().sum())
print("\nPercentage of Missing Values:")
print((df.isnull().sum() / len(df)) * 100)

In [ ]:
# Target distribution overview
target_counts = df['Heart_Disease'].value_counts()
print("Target Distribution (Heart_Disease):")
print(target_counts)

plt.figure(figsize=(6, 4))
sns.countplot(x='Heart_Disease', data=df, hue='Heart_Disease', palette='Set2', legend=False)
plt.title('Heart Disease Target Distribution (0: No Disease, 1: Disease)')
plt.ylabel('Count')
plt.show()

## 3. Exploratory Data Analysis & Visualizations
We will plot numerical feature distributions and correlations to see how variables affect heart disease status.

In [ ]:
# 3.1 Age Distribution by Heart Disease Status
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Age', hue='Heart_Disease', kde=True, multiple='stack', palette='coolwarm')
plt.title('Age Distribution by Heart Disease Status')
plt.xlabel('Age (Years)')
plt.ylabel('Count')
plt.show()

In [ ]:
# 3.2 Cholesterol Distribution by Heart Disease Status
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Cholesterol', hue='Heart_Disease', kde=True, multiple='stack', palette='coolwarm')
plt.title('Cholesterol Distribution by Heart Disease Status')
plt.xlabel('Serum Cholesterol (mg/dl)')
plt.ylabel('Count')
plt.show()

In [ ]:
# 3.3 Correlation Heatmap of Numerical Features
numeric_cols = ["Age", "Resting_Blood_Pressure", "Cholesterol", "Maximum_Heart_Rate", "Oldpeak", "Heart_Disease"]
corr_matrix = df[numeric_cols].dropna().corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

## 4. Reusable Preprocessing Pipeline
Let's load the data preprocessing pipeline from `src/train.py`. This pipeline handles missing values using median imputation for numerical data and most-frequent imputation for categorical data, encodes category labels, scales variables, and performs train-test splitting.

In [ ]:
from train import preprocess_data

# Preprocess and split the dataset
X_train, X_test, y_train, y_test, preprocessor = preprocess_data(df)
print("Train-Test sets and preprocessor initialized successfully.")

## 5. Model Training and Comparison
We train 4 classifiers (Logistic Regression, Decision Tree, Random Forest, SVM) and evaluate their performance. Let's run the training and comparison logic.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from train import calculate_metrics

# Transform features using the preprocessor
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Initialize classifiers
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6),
    "Support Vector Machine": SVC(probability=True, random_state=42, kernel='rbf')
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_processed, y_train)
    y_pred = clf.predict(X_test_processed)
    y_prob = clf.predict_proba(X_test_processed)[:, 1]
    metrics = calculate_metrics(y_test.values, y_pred, y_prob)
    results[name] = metrics

comparison_df = pd.DataFrame(results).T
comparison_df.round(4)

### 5.1 Visualize Model Comparison
Let's visualize the performance comparison across classifiers.

In [ ]:
df_melted = comparison_df.reset_index().melt(
    id_vars="index", var_name="Metric", value_name="Score"
).rename(columns={"index": "Model"})

plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Score", hue="Metric", data=df_melted, palette="Set2")
plt.title("Classifier Metrics Comparison")
plt.ylim([0.0, 1.1])
plt.ylabel("Score (0.0 - 1.0)")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

## 6. Detailed Evaluation of the Best Model
Based on our metrics, we evaluate the best-performing model (Support Vector Machine) in detail by displaying its Confusion Matrix and ROC Curve.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

best_model_name = comparison_df["F1_Score"].idxmax()
best_clf = models[best_model_name]
print(f"Best model is: {best_model_name}\n")

y_pred_best = best_clf.predict(X_test_processed)
y_prob_best = best_clf.predict_proba(X_test_processed)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred_best))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Disease', 'Disease'], yticklabels=['No Disease', 'Disease'], cbar=False)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
auc_val = roc_auc_score(y_test, y_prob_best)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_val:.3f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve - {best_model_name}')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Random Forest Feature Importance
from train import clean_feature_names
rf_model = models["Random Forest"]
importances = rf_model.feature_importances_
raw_names = preprocessor.get_feature_names_out()
cleaned_names = clean_feature_names(raw_names)

feat_df = pd.DataFrame({
    "Feature": cleaned_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x="Importance", y="Feature", data=feat_df.head(10), palette="viridis")
plt.title("Top 10 Feature Importances (Random Forest Model)")
plt.xlabel("Relative Importance")
plt.ylabel("Feature")
plt.show()

## 7. Conclusions
- We successfully generated a realistic cardiac dataset and implemented a pipeline.
- Preprocessing includes median imputation for numerical data and most-frequent imputation for categorical variables, scaling, and one-hot encoding.
- SVM and Logistic Regression show excellent, reliable clinical classification metrics on this dataset.
- This model is saved in `../model/heart_disease_model.pkl` for serving in the Streamlit web dashboard and CLI predictions.